# Intro til Machine Learning — 4: Ekstra opgaver

Her er der ekstra at rode med, når I er foran: først en række store, kombinerede udfordringer, og til sidst en hel værktøjskasse mod overfitting (på et nyt datasæt).


In [ ]:
!pip install -q kagglehub

!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/Pokemon.csv
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_test_lille.csv.gz

import kagglehub   # bruges kun i E.8 (frit valg fra Kaggle)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from helpers import show_mnist_images

torch.manual_seed(42)
np.random.seed(42)

# Pokémon (hentet med wget ovenfor)
df = pd.read_csv("Pokemon.csv")

# MNIST (bruges i E.4, E.6 og E.7) — allerede nedskaleret
train_df = pd.read_csv("mnist_traen_lille.csv.gz").sample(n=8000, random_state=42).reset_index(drop=True)
test_df = pd.read_csv("mnist_test_lille.csv.gz").reset_index(drop=True)

X_mnist = torch.tensor(train_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist = torch.tensor(train_df["label"].values, dtype=torch.long)
X_mnist_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print("klar!", df.shape, X_mnist.shape)

# 1: Store udfordringer

Vælg frit — de kan tages i vilkårlig rækkefølge.


##### Opgave 1.1 — Pokédex-rapporten

I er blevet hyret som dataanalytikere af Professor Oak. Han vil vide: **hvilken generation
er den "bedste"?** Men "bedst" er jeres valg at definere — stærkest i snit? flest
legendariske? bedst balance mellem angreb og forsvar? mest variation?

**Krav til rapporten (byg den i denne notebook):**
1. Mindst **3 forskellige plots** (fx histogram, scatter, søjler)
2. Mindst én **groupby**
3. Mindst én **ny kolonne**, I selv har beregnet
4. En **konklusion i en markdown-celle**: hvilken generation vinder, og hvorfor?

Startkoden giver et første fingerpeg — resten er op til jer.

In [ ]:
# Første fingerpeg: gennemsnitlig Total pr. generation
df.groupby("Generation")["Total"].mean().plot(kind="bar")
plt.ylabel("gennemsnitlig Total")
plt.title("Er nyere generationer stærkere?")
plt.show()

# jeres analyse fortsætter her...

##### Opgave 1.2 — Gradient descent i 2D

I regressionsforløbet rullede I ned ad en 1D-bakke. Rigtige tabslandskaber har millioner af
dimensioner — men i 2D kan vi stadig TEGNE det. Funktionen
$f(a, b) = (a-1)^2 + 3(b+2)^2$ er en oval dal med bund i $(1, -2)$.

**Jeres mission:** udfyld gradienten og opdateringsskridtene, og tegn ruten ned i dalen
oven på konturplottet. Prøv derefter forskellige startpunkter og læringsrater. (Bonus:
brug autograd i stedet for hånd-gradienten og tjek, at ruten er den samme.)

In [ ]:
def f(a, b):
    return (a - 1) ** 2 + 3 * (b + 2) ** 2

# Konturplottet ("landkortet" — hver ring er en højdekurve):
A, B = np.meshgrid(np.linspace(-4, 6, 100), np.linspace(-6, 2, 100))
plt.contour(A, B, f(A, B), levels=30, cmap="viridis")
plt.colorbar(label="f(a, b)")

# Gradient descent — udfyld de to partielle afledede og opdateringen:
a, b = 5.0, 1.0                  # startpunkt
learning_rate = 0.1
route_a, route_b = [a], [b]
for i in range(40):
    grad_a = ...                 # ∂f/∂a = 2(a - 1)
    grad_b = ...                 # ∂f/∂b = 6(b + 2)
    a = ...
    b = ...
    route_a.append(a)
    route_b.append(b)

plt.plot(route_a, route_b, "o-", color="crimson", markersize=4)
plt.scatter([1], [-2], color="gold", s=150, zorder=3, edgecolors="black", label="minimum")
plt.legend()
plt.title(f"GD ender i ({a:.2f}, {b:.2f})")
plt.show()

##### Opgave 1.3 — Type-oraklet

Kan man se på en Pokémons stats, hvilken **type** den er? Det er klassifikation med
**18 klasser** — jeres hidtil vildeste. Udfyld hullerne (mønstret er 12.6 + notebook 4).

Bagefter: sammenlign med den dovne baseline (gæt altid på den mest almindelige type —
hvor god er DEN? Husk opgave 2.5!), og diskutér: hvorfor er det her SÅ meget sværere end
at spotte legendariske?

In [ ]:
types = sorted(df["Type 1"].unique())
print(len(types), "typer:", types)
type_til_nr = {t: nr for nr, t in enumerate(types)}

seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)
y_np = df["Type 1"].map(type_til_nr).values

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train, X_test = torch.tensor(X_train_np), torch.tensor(X_test_np)
y_train = torch.tensor(y_train_np, dtype=...)        # ← hvilken dtype vil CrossEntropy have?
y_test = torch.tensor(y_test_np, dtype=torch.long)

class TypeOracle(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 64)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(64, ...)               # ← hvor mange klasser?

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

model = TypeOracle()
loss_fn = ...                                    # ← tabsfunktionen?
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
for epoch in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
print(f"oraklets accuracy:  {(pred == y_test).float().mean().item():.1%}")

# den dovne baseline: gæt ALTID på den mest almindelige type
...

##### Opgave 1.4 — Den store aktiveringsdyst

Afgør det én gang for alle (i hvert fald for MNIST): træn det SAMME netværk med **ReLU,
Sigmoid, Tanh og LeakyReLU** og vis de fire test-accuracies i ét søjlediagram. Skabelonen
har allerede løkken — I skal udfylde trænings-rytmen og accuracy-målingen (mønstret fra
notebook 4).

In [ ]:
results = {}

for name, akt in [("ReLU", nn.ReLU()), ("Sigmoid", nn.Sigmoid()),
                  ("Tanh", nn.Tanh()), ("LeakyReLU", nn.LeakyReLU())]:
    torch.manual_seed(42)                       # fair kamp: samme startvægte
    model = nn.Sequential(nn.Linear(784, 128), akt, nn.Linear(128, 10))
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(3):
        for i in range(0, len(X_mnist), 64):
            ...                                  # ← trænings-rytmen (4 linjer)

    with torch.no_grad():
        acc = ...                                # ← test-accuracy (argmax!)
    results[name] = acc
    print(f"{name}: {acc:.1%}")

plt.bar(results.keys(), results.values())
plt.ylim(0.8, 1.0)
plt.ylabel("test-accuracy")
plt.title("Aktiveringsdysten (3 epoker)")
plt.show()

##### Opgave 1.5 — Hyperparameter-jagten (ekstra)

Læringsrate og netværksstørrelse kaldes **hyperparametre** — tal, VI vælger, og som
modellen ikke selv lærer. I praksis prøver man sig systematisk frem. Nu skal pandas møde
PyTorch: kør en **dobbelt for-løkke** over læringsrater × skjulte størrelser (på et udsnit
af MNIST, så det går hurtigt), gem alle resultater i en **DataFrame**, og vis dem som et
**heatmap**. Udfyld hullerne — og find den bedste kombination!

In [ ]:
X_lille = X_mnist[:3000]
y_lille = y_mnist[:3000]

results = []
for lr in [0.0001, 0.001, 0.01]:
    for hidden in [16, 64, 256]:
        torch.manual_seed(42)
        model = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, 10))
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        for epoch in range(2):
            for i in range(0, len(X_lille), 64):
                ...                              # ← rytmen igen (4 linjer)

        with torch.no_grad():
            acc = (model(X_mnist_test).argmax(dim=1) == y_mnist_test).float().mean().item()
        results.append({"lr": lr, "skjulte": hidden, "accuracy": acc})
        print(f"lr={lr}, skjulte={hidden}: {acc:.1%}")

res_df = pd.DataFrame(results)
table = res_df.pivot(index="lr", columns="skjulte", values="accuracy")
print(table)

# heatmap af tabellen (imshow-mønstret fra notebook 1):
plt.imshow(table, cmap="viridis")
plt.colorbar(label="accuracy")
plt.xticks(range(len(table.columns)), table.columns)
plt.yticks(range(len(table.index)), table.index)
plt.xlabel("skjulte neuroner")
plt.ylabel("læringsrate")
plt.title("Hyperparameter-jagten")
plt.show()

##### Opgave 1.6 — Tegn selv et tal

Kan jeres netværk læse JERES håndskrift? Et 28×28-billede er bare et numpy-array, så I kan
"tegne" med slicing: `billede[raekker, kolonner] = 1.0` maler et hvidt felt. Startkoden
tegner et (meget kantet) 7-tal og spørger modellen.

**Jeres mission:** tegn mindst to andre cifre med slicing, og se om modellen kan læse dem.
Kig på sandsynlighederne — hvornår er den i tvivl? (Og hvis den fejler: er det jeres
håndskrift eller modellens skyld? )

In [ ]:
# først: et hurtigtrænet netværk (samme opskrift som før)
torch.manual_seed(42)
model_e6 = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
optimizer = torch.optim.Adam(model_e6.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(5):
    for i in range(0, len(X_mnist), 64):
        optimizer.zero_grad()
        loss = loss_fn(model_e6(X_mnist[i:i + 64]), y_mnist[i:i + 64])
        loss.backward()
        optimizer.step()
print("model klar!")

# tegn et 7-tal med slicing — RET I DET HER og tegn jeres egne cifre:
image = np.zeros((28, 28), dtype=np.float32)
image[5:8, 6:21] = 1.0       # vandret streg foroven
image[8:23, 17:20] = 1.0     # lodret streg ned

plt.imshow(image, cmap="gray")
plt.show()

# hvad siger modellen?
with torch.no_grad():
    point = model_e6(torch.tensor(image.reshape(1, 784)))
probabilities = torch.softmax(point, dim=1).squeeze()
print("modellens gæt:", point.argmax().item())
plt.bar(range(10), probabilities)
plt.xticks(range(10))
plt.title("modellens sandsynligheder")
plt.show()

##### Opgave 1.7 — Fejl-detektiven (boss-level)

Pipeline-koden nedenfor skal træne en legendarisk-spotter (som notebook 2) — men der
gemmer sig **FEM fejl** fra hele emnet i den. Nogle crasher med en fejlbesked, andre
snyder i **total stilhed**. Find og ret alle fem!

*Tip: I har mødt hver eneste af fejlene før. Tjek: dataforberedelsen, modellens output,
loop-rytmen (×2) — og til sidst: hvad der egentlig bliver målt.*

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
y_np = df["Legendary"].astype(int).values.astype("float32")

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train, X_test = torch.tensor(X_train_np), torch.tensor(X_test_np)
y_train, y_test = torch.tensor(y_train_np), torch.tensor(y_test_np)

class Spotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

model = Spotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(500):
    y_hat = model(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    optimizer.step()
    loss.backward()

with torch.no_grad():
    pred = (model(X_train).squeeze() > 0.5).float()
accuracy = (pred == y_train).float().mean()
print(f"accuracy: {accuracy.item():.1%}")

##### Opgave 1.8 — Frit valg fra Kaggle (ekstra)

Den sidste boss er friheden selv: **find jeres eget datasæt på
[kaggle.com/datasets](https://www.kaggle.com/datasets)** og træn et netværk på det.

**Tjekliste (= alt hvad I har lært):**
1. Find et datasæt med en CSV-fil og mindst én kolonne, der er værd at forudsige.
 Gode begynder-emner: vin-kvalitet, bilpriser, fisk-vægt, film-ratings...
2. Hent det: `kagglehub.dataset_download("ejer/datasæt-navn")` (navnet står i datasættets URL).
3. Udforsk og rens: `head`, `describe`, `isna().sum()` — drop eller udfyld huller,
 og vælg talkolonner som features.
4. Standardisér features. Train/test-split.
5. Regression eller klassifikation? → vælg output-lag og tabsfunktion (opgave 8.8!).
6. Byg model, træn med rytmen, plot tabskurven.
7. Mål på testsættet — og sammenlign ALTID med en doven baseline (gennemsnittet /
 den største klasse).

Skabelonen nedenfor er jeres startgerüst — udfyld den trin for trin.

In [ ]:
# 1-2: hent jeres datasæt
path_own = kagglehub.dataset_download("...")     # ← udfyld
import os
print(os.listdir(path_own))                       # hvilke filer kom der?

own_df = pd.read_csv(path_own + "/....csv")      # ← udfyld filnavnet
own_df.head()

In [ ]:
# Henter hjerte-data fra GitHub (Plan B: upload heart.csv via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/heart.csv

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("heart.csv")
df.head()

Hver række er en patient: alder, køn, blodtryk, kolesterol, maks-puls, brystsmerte-type
m.m. — og `HeartDisease` (1 = hjertesygdom). Tekstkolonnerne one-hot-koder vi med
`pd.get_dummies` (hver kategori får sin egen 0/1-kolonne), og så standardiserer vi,
som vi plejer:

In [ ]:
X = pd.get_dummies(df.drop(columns=["HeartDisease"])).astype("float32")
print("features efter one-hot:", X.shape)

X_np = X.values
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)     # standardisering, som altid
y_np = df["HeartDisease"].values.astype("float32")
print("andel med hjertesygdom:", round(float(y_np.mean()), 3))

# 2: Valideringssættet — den manglende brik

I Intro-ML delte vi data i **train** og **test**. Men der er et hul i den plan: når I
skruer på epoker, læringsrate og netstørrelse og måler på testsættet hver gang — så
vælger I til sidst præcis de indstillinger, der tilfældigvis passer TESTSÆTTET. I har
"slidt eksamen op": modellen består en prøve, den reelt har fået lov at øve sig på.

Derfor deler professionelle ALTID i tre:

| Sæt | Rolle | Skole-analogi |
|---|---|---|
| **Træning** (60 %) | modellen lærer af det | lektier |
| **Validering** (20 %) | VI vælger indstillinger efter det | terminsprøver |
| **Test** (20 %) | røres ÉN gang, til allersidst | eksamen |

Splittet laver vi med `train_test_split` to gange — først 60/40, så deles de 40 i to:

In [ ]:
X_train_np, X_rest, y_train_np, y_rest = train_test_split(X_np, y_np, test_size=0.4, random_state=42)
X_val_np, X_test_np, y_val_np, y_test_np = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)

X_train, X_val, X_test = map(torch.tensor, (X_train_np, X_val_np, X_test_np))
y_train, y_val, y_test = map(torch.tensor, (y_train_np, y_val_np, y_test_np))

print("træning:", len(X_train), "| validering:", len(X_val), "| test:", len(X_test))

## Fremprovokér overfitting!

Nu gør vi bevidst alt det, man ikke må: et KÆMPE netværk (over 70.000 parametre) på
BITTESMÅ data (550 patienter), trænet længe. Undervejs noterer vi tabet på både
træningsdata og valideringsdata — valideringstabet beregnes i `torch.no_grad()`, for
modellen må gerne blive MÅLT på valideringssættet, men aldrig LÆRE af det:

In [ ]:
def build_net():
    return nn.Sequential(
        nn.Linear(X.shape[1], 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 1), nn.Sigmoid())

model = build_net()
print("parametre:", sum(p.numel() for p in model.parameters()))

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
train_loss, val_loss = [], []

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()

    with torch.no_grad():                          # mål, men lær ikke!
        train_loss.append(loss.item())
        val_loss.append(loss_fn(model(X_val).squeeze(), y_val).item())

In [ ]:
plt.plot(train_loss, label="træningstab")
plt.plot(val_loss, label="valideringstab")
plt.xlabel("epoke")
plt.ylabel("BCE-tab")
plt.legend()
plt.title("Den vigtigste figur i praktisk deep learning")
plt.show()

print(f"laveste valideringstab: {min(val_loss):.3f} ved epoke {val_loss.index(min(val_loss))}")

**DÉR er overfitting.** Træningstabet falder og falder (modellen lærer patienterne
udenad) — men valideringstabet bunder omkring epoke ~130 og **stiger derefter**. Fra
det punkt bliver modellen bedre til at HUSKE og dårligere til at FORUDSIGE. Al træning
efter bunden er ikke bare spildt — den er skadelig.

## Early stopping: stop, mens legen er god

Det oplagte modtræk: gem modellen, hver gang valideringstabet sætter ny bundrekord —
og brug til sidst den gemte udgave i stedet for den sidste:

In [ ]:
torch.manual_seed(42)
model = build_net()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

best_val_loss = 999.0
best_epoch = 0
best_weights = None

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        val_now = loss_fn(model(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:                              # ny bundrekord?
        best_val_loss = val_now
        best_epoch = epoch
        best_weights = copy.deepcopy(model.state_dict())    # gem en KOPI af vægtene

model.load_state_dict(best_weights)                         # spol tilbage til rekorden
print(f"bedste model: epoke {best_epoch} (valideringstab {best_val_loss:.3f})")

with torch.no_grad():
    val_acc = ((model(X_val).squeeze() > 0.5).float() == y_val).float().mean()
print(f"validerings-accuracy: {val_acc.item():.1%}")

(`model.state_dict()` er en ordbog med alle modellens vægte, og `copy.deepcopy` tager
en RIGTIG kopi — ellers ville vores "gemte" vægte bare pege på de levende vægte, der
ændrer sig videre. `load_state_dict` lægger dem tilbage.)

### Opgaver

##### Opgave 2.1
Aflæs U-kurven fra det store eksperiment: cirka ved hvilken epoke begynder
valideringstabet at stige? Og hvad er der galt med at sige *"men træningstabet falder
jo stadig — modellen bliver da bedre!"*?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 2.2
Hvorfor kan vi ikke bare bruge TESTSÆTTET til at vælge antal epoker (og droppe
valideringssættet)? Hvad ville der ske med troværdigheden af vores allersidste
test-måling?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 2.3
Udfyld de to `test_size`-værdier, så splittet bliver præcis 60/20/20. (Tænk: andet
split deler RESTEN — hvor stor en andel af de 40 % skal blive til test?)

In [ ]:
X_a, X_r, y_a, y_r = train_test_split(X_np, y_np, test_size=..., random_state=42)
X_v, X_t, y_v, y_t = train_test_split(X_r, y_r, test_size=..., random_state=42)
print(len(X_a), len(X_v), len(X_t))   # skal give ca. 550 / 184 / 184

##### Opgave 2.4
Gentag overfitting-eksperimentet med et LILLE net: skift alle 256-taller ud med 16.
Hvordan ser kurverne ud nu — og hvorfor giver det mening, at et lille net overfitter
mindre?

In [ ]:
torch.manual_seed(42)
model_lille = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(),     # ← 256 → 16 alle tre steder
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())
optimizer = torch.optim.Adam(model_lille.parameters(), lr=0.0001)
train_loss2, val_loss2 = [], []
for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_lille(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        train_loss2.append(loss.item())
        val_loss2.append(loss_fn(model_lille(X_val).squeeze(), y_val).item())

plt.plot(train_loss2, label="træningstab")
plt.plot(val_loss2, label="valideringstab")
plt.legend()
plt.show()

##### Opgave 2.5 (find fejlen)
Denne kammerats "valideringskurve" ser mistænkeligt godt ud — den falder bare og
falder, intet U! Kig godt på linjen, hvor valideringstabet beregnes. Hvad er der sket —
og hvorfor er netop DEN fejl så farlig? (Den giver jo ingen fejlbesked...)

In [ ]:
torch.manual_seed(42)
model_k = build_net()
optimizer = torch.optim.Adam(model_k.parameters(), lr=0.0001)
val_loss_k = []
for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_k(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        val_loss_k.append(loss_fn(model_k(X_train).squeeze(), y_train).item())

plt.plot(val_loss_k, label="'valideringstab'")
plt.legend()
plt.show()

##### Opgave 2.6
Overfitting elsker små datasæt. Brug kun de første 275 træningspatienter
(`X_traen[:275]`, `y_traen[:275]`) og kør det store net igen. Kommer U-kurvens bund
tidligere eller senere — og bliver gabet mellem kurverne større eller mindre?

In [ ]:
torch.manual_seed(42)
model_h = build_net()
optimizer = torch.optim.Adam(model_h.parameters(), lr=0.0001)
train_loss3, val_loss3 = [], []
for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_h(X_train).squeeze(), y_train)      # ← brug kun de første 275!
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        train_loss3.append(loss.item())
        val_loss3.append(loss_fn(model_h(X_val).squeeze(), y_val).item())

plt.plot(train_loss3, label="træningstab")
plt.plot(val_loss3, label="valideringstab")
plt.legend()
plt.show()

##### Opgave 2.7
Udfyld de tre manglende linjer i early stopping (gem rekorden, epoken og en KOPI af
vægtene).

In [ ]:
torch.manual_seed(42)
model_es = build_net()
optimizer = torch.optim.Adam(model_es.parameters(), lr=0.0001)
best_val_loss = 999.0
best_epoch = 0
best_weights = None

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_es(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        val_now = loss_fn(model_es(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:
        best_val_loss = ...
        best_epoch = ...
        best_weights = ...

model_es.load_state_dict(best_weights)
print(f"bedste epoke: {best_epoch} (val-tab {best_val_loss:.3f})")

##### Opgave 2.8 (ekstra)
Rigtig early stopping venter ikke til epoke 600 — den AFBRYDER, når valideringstabet
ikke har sat bundrekord i `taalmodighed` epoker i træk. Udfyld tæller-logikken
(nulstil ved rekord, tæl op ellers). Hvor mange epoker sparede I?

In [ ]:
torch.manual_seed(42)
model_p = build_net()
optimizer = torch.optim.Adam(model_p.parameters(), lr=0.0001)
best_val_loss = 999.0
patience = 50
siden_rekord = 0

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_p(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        val_now = loss_fn(model_p(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:
        best_val_loss = val_now
        siden_rekord = ...          # rekord! nulstil tælleren
    else:
        siden_rekord = ...          # ingen rekord — tæl op
    if siden_rekord >= patience:
        print(f"early stopping ved epoke {epoch}!")
        break

# 3: Værktøjskassen — dropout, weight decay & batch norm

Early stopping behandler symptomet (stop før udenadslæren). De næste værktøjer angriber
årsagen: de gør det SVÆRERE for nettet at lære udenad.

For ikke at skrive det samme loop fem gange pakker vi det i en funktion — læs den, det
er jeres velkendte rytme plus én ny detalje: **`model.train()` og `model.eval()`**.
De to kald tænder og slukker for "træningsopførsel". Hvorfor det pludselig er vigtigt,
ser I om lidt:

In [ ]:
def train_with_curves(model, epochs=600, lr=0.0001, weight_decay=0.0):
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loss, val_loss = [], []
    for epoch in range(epochs):
        model.train()                                # træningsopførsel TIL
        optimizer.zero_grad()
        loss = loss_fn(model(X_train).squeeze(), y_train)
        loss.backward()
        optimizer.step()

        model.eval()                                 # træningsopførsel FRA, før vi måler
        with torch.no_grad():
            train_loss.append(loss_fn(model(X_train).squeeze(), y_train).item())
            val_loss.append(loss_fn(model(X_val).squeeze(), y_val).item())
    return train_loss, val_loss


def val_accuracy(model):
    model.eval()
    with torch.no_grad():
        return ((model(X_val).squeeze() > 0.5).float() == y_val).float().mean().item()

## Dropout: træn med huller i holdet

**Dropout** slukker i hvert træningsskridt en tilfældig andel af neuronerne (fx 50 %).
Så kan ingen enkelt neuron få lov at huske hele facitlisten — holdet tvinges til at
sprede viden ud. Som fodboldtræning, hvor tilfældige holdkammerater udebliver hver
gang: alle ender med at kunne alle positioner.

Vigtigt: hullerne er KUN til træning. Når modellen skal bruges, spiller hele holdet —
det er derfor `model.eval()` findes. I PyTorch er dropout bare et lag:

In [ ]:
torch.manual_seed(42)
model_dropout = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())

train_loss_d, val_loss_d = train_with_curves(model_dropout)

plt.plot(val_loss, label="uden dropout (fra afsnit 3)")
plt.plot(val_loss_d, label="med dropout 0.5")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()
print(f"validerings-accuracy med dropout: {val_accuracy(model_dropout):.1%}")

## Weight decay: hold vægtene små

Udenadslære kræver typisk store, ekstreme vægte ("HVIS præcis denne kombination, SÅ
syg!"). **Weight decay** lægger en lille straf på store vægte oveni tabet, så nettet
foretrækker små vægte — og dermed enkle, glatte løsninger. I PyTorch er det
bogstaveligt talt ét ekstra argument til optimizeren:

In [ ]:
torch.manual_seed(42)
model_wd = build_net()
train_loss_w, val_loss_w = train_with_curves(model_wd, weight_decay=0.01)

plt.plot(val_loss, label="uden noget")
plt.plot(val_loss_w, label="med weight decay 0.01")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()
print(f"validerings-accuracy med weight decay: {val_accuracy(model_wd):.1%}")

## Batch normalization: stabilisatoren

**Batch norm** standardiserer tallene MELLEM lagene (jeres standardiserings-trick,
bare gentaget inde i nettet). Det gør træning af DYBE netværk markant mere stabil og
hurtig, og næsten alle store moderne net bruger det.

Men lad os være ærlige og AFPRØVE det i stedet for at tro på reklamen:

In [ ]:
torch.manual_seed(42)
model_bn = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())

train_loss_b, val_loss_b = train_with_curves(model_bn)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_loss, label="uden")
axes[0].plot(train_loss_b, label="med batch norm")
axes[0].set_title("træningstab")
axes[0].legend()
axes[1].plot(val_loss, label="uden")
axes[1].plot(val_loss_b, label="med batch norm")
axes[1].set_title("valideringstab")
axes[1].legend()
plt.show()

Se godt efter: med batch norm **styrtdykker træningstabet** (stabilisatoren virker!)
— men valideringstabet vender TIDLIGERE og stiger STEJLERE. På et lillebitte datasæt
som vores accelererer batch norm bare udenadslæren. Lærdommen er vigtig: batch norm er
et værktøj til at få STORE, DYBE net til at træne stabilt — det er ikke et middel mod
overfitting. Kend jeres værktøjs job.

## Den samlede opskrift

I praksis kombinerer man: et fornuftigt net + dropout + lidt weight decay + early
stopping. Lad os køre hele pakken — og til allersidst, én gang, røre testsættet:

In [ ]:
torch.manual_seed(42)
model_fuld = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_fuld.parameters(), lr=0.0001, weight_decay=0.01)
best_val_loss = 999.0
best_weights = None

for epoch in range(600):
    model_fuld.train()
    optimizer.zero_grad()
    loss = loss_fn(model_fuld(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()

    model_fuld.eval()
    with torch.no_grad():
        val_now = loss_fn(model_fuld(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:
        best_val_loss = val_now
        best_weights = copy.deepcopy(model_fuld.state_dict())

model_fuld.load_state_dict(best_weights)
model_fuld.eval()
with torch.no_grad():
    test_acc = ((model_fuld(X_test).squeeze() > 0.5).float() == y_test).float().mean()
print(f"ENDELIG test-accuracy (målt én gang!): {test_acc.item():.1%}")

| Værktøj | Hvad gør det? | Hvornår? |
|---|---|---|
| Valideringssæt | ærlig måling undervejs | ALTID |
| Early stopping | stop før udenadslæren | næsten altid |
| Dropout | sluk neuroner → tving samarbejde | mellemstore/store net |
| Weight decay | straf store vægte → enkle løsninger | næsten altid (små doser) |
| Batch norm | stabilisér træning af DYBE net | dybe net, store datasæt |
| Mindre net / mere data | angrib årsagen direkte | når muligt! |

### Opgaver

##### Opgave 3.1
Prøv dropout på **0.1, 0.5 og 0.9** og sammenlign valideringskurverne i ét plot. Hvad
går der galt ved 0.9 — og hvad hedder det modsatte af overfitting?

In [ ]:
for rate in [0.1]:      # ← prøv [0.1, 0.5, 0.9]
    torch.manual_seed(42)
    m = nn.Sequential(
        nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(rate),
        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(rate),
        nn.Linear(256, 1), nn.Sigmoid())
    t_loss, v_loss = train_with_curves(m)
    plt.plot(v_loss, label=f"dropout {rate}")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()

##### Opgave 3.2 (find fejlen)
En kammerat måler accuracy på dropout-modellen — og får et NYT tal, hver gang cellen
køres?! Og tallene er lavere end forventet. Kør cellen 3-4 gange og se selv. Én linje
er forkert — hvilken, og hvorfor ændrer den alt?

In [ ]:
model_dropout.train()   # ... den har vel ikke noget at sige?
with torch.no_grad():
    acc = ((model_dropout(X_val).squeeze() > 0.5).float() == y_val).float().mean()
print(f"accuracy: {acc.item():.1%}   — kør cellen igen: nyt tal!")

##### Opgave 3.3
Udfyld `weight_decay`-argumentet (prøv 0.01), og tjek at valideringskurven bliver
fladere end uden.

In [ ]:
torch.manual_seed(42)
m = build_net()
optimizer = torch.optim.Adam(m.parameters(), lr=0.0001, ...)
# genbrug evt. traen_med_kurver i stedet — den tager weight_decay som parameter
t_loss, v_loss = train_with_curves(m, weight_decay=...)
plt.plot(val_loss, label="uden")
plt.plot(v_loss, label="med weight decay")
plt.legend()
plt.show()

##### Opgave 3.4
Weight decay har også en overdosis: prøv **0, 0.0001, 0.01 og 1.0** og sammenlign
kurver/accuracy. Hvad sker der ved 1.0, og hvorfor?

In [ ]:
for wd in [0.0]:      # ← prøv [0, 0.0001, 0.01, 1.0]
    torch.manual_seed(42)
    m = build_net()
    t_loss, v_loss = train_with_curves(m, weight_decay=wd)
    plt.plot(v_loss, label=f"wd = {wd}")
plt.legend()
plt.show()

##### Opgave 3.5
Modellen skal bruges på rigtige patienter. Diskutér: hvad koster en **falsk negativ**
(syg patient, modellen siger rask) i forhold til en **falsk positiv** (rask patient,
modellen siger syg)? Og hvad kunne man gøre ved 0,5-grænsen, hvis man hellere vil have
den ene slags fejl end den anden?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 3.6
Byg jeres bedste opskrift: kombinér valgfrit netværksstørrelse, dropout, weight decay
og early stopping — men mål KUN på valideringssættet, mens I eksperimenterer. Når I har
valgt jeres endelige opskrift: mål på testsættet ÉN gang. Hvor tæt ligger jeres
validerings- og test-accuracy?

In [ ]:
# eksperimentér her (mål kun på validering!) — og til allersidst: én test-måling
torch.manual_seed(42)
m = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())
t_loss, v_loss = train_with_curves(m, weight_decay=0.01)
print(f"validering: {val_accuracy(m):.1%}")

# NÅR (og kun når) I er færdige med at eksperimentere:
# m.eval()
# with torch.no_grad():
#     test_acc = ((m(X_test).squeeze() > 0.5).float() == y_test).float().mean()
# print(f"test: {test_acc.item():.1%}")

##### Opgave 3.7 (ekstra)
Batch norm fik hårde ord ovenfor — men den lovede også HURTIGERE træning. Mål det:
hvor mange epoker tager det henholdsvis med og uden batch norm at få træningstabet
under 0.3? (Skriv en for-løkke, der finder den første epoke under grænsen.)

In [ ]:
torch.manual_seed(42)
m_uden = build_net()
t_uden, v_uden = train_with_curves(m_uden)

torch.manual_seed(42)
m_med = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())
t_med, v_med = train_with_curves(m_med)

# find første epoke hvor tabet er under 0.3 — for begge:
...

##### Opgave 3.8 (ekstra)
Tilbage til Intro-ML-tricket: gennemsnittet af en True/False-række. Beregn hvor mange af
testsættets FAKTISK SYGE patienter jeres endelige model fanger (recall) — og sammenlign
med den samlede accuracy. Hvilket af de to tal ville I vise til en læge først?

In [ ]:
model_fuld.eval()
with torch.no_grad():
    pred = (model_fuld(X_test).squeeze() > 0.5).float()

sick = y_test == 1
recall = ...          # andelen af de syge, som modellen fanger
print(f"accuracy: {(pred == y_test).float().mean().item():.1%}")
print(f"recall:   {recall:.1%}")